In [1]:
from typing import Union
import numpy as np

In [2]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 100)
def read_outputs(file_path):
    outputs = pd.read_json(file_path, lines=True)
    outputs = outputs.explode('generations',ignore_index=True)
    # outputs['generations'] = outputs['generations'].apply(lambda x: [x])
    outputs['prompt']=outputs['prompt'].apply(lambda x: x['text'])
    
    # outputs['text']=outputs['generations'].apply(lambda x: x['text'])
    
    gen_dict=outputs['generations'].values[0]
    
    for col in gen_dict.keys():
        outputs[col] = outputs['generations'].apply(lambda x: x.get(col,None))
        
    outputs.drop(columns=['generations'],inplace=True)
    return outputs

def read_int_outputs(file_path):
    outputs = pd.read_json(file_path, lines=True)
    outputs = outputs.explode('generations',ignore_index=True)
    # outputs['generations'] = outputs['generations'].apply(lambda x: [x])
    outputs['prompt']=outputs['prompt'].apply(lambda x: x['text'])
    
    # outputs['text']=outputs['generations'].apply(lambda x: x['text'])
    
    gen_dict=outputs['generations'].values[0]
    
    for col in gen_dict.keys():
        outputs[col] = outputs['generations'].apply(lambda x: x.get(col,None))
        
    outputs.drop(columns=['generations'],inplace=True)
    return outputs

def ravel(unraveled_df):
    if 'tokens' in unraveled_df:
        unraveled_df['generations']= unraveled_df.apply(lambda x: [{'text': x['text'],
                                                               'tokens': x['tokens']}],axis=1)
    else:
        unraveled_df['generations']= unraveled_df.apply(lambda x: [{'text': x['text']}],axis=1)
    return_df = unraveled_df.groupby('prompt')['generations'].sum([]).reset_index()
    return_df['prompt'] = return_df['prompt'].apply(lambda x: {'text':x})
        
    return return_df

In [12]:
def read_metric_file(result_file, metric) -> Union[np.array, pd.DataFrame]:
        
    if metric == 'ppl-big-qwen':
        result=pd.read_csv(result_file,header=None)
        return result
    
    elif metric == 'fluency':
        result=pd.read_csv(result_file,header=None)
        return result[0].apply(lambda x: 1 if x=='LABEL_1' else 0).values
    
    elif metric == 'repetitions':
        result=pd.read_json(result_file,lines=True)
        if result.empty:
            return np.zeros(result.shape[0])
        else:
            return (~result['repeated_phrase'].isna()).apply(lambda x: 1 if x==True else 0).values
        
    elif metric == 'toxicity':
        result=pd.read_json(result_file,lines=True)
        return unravel_toxicity_data(result)
    
    elif metric == 'sentiment_ext':
        result=pd.read_json(result_file,lines=True)
        return result['label'].apply(lambda x: 1 if x == 'POSITIVE' else 0).values
    
    elif metric == 'formality_ext':
        result = pd.read_csv(result_file,header=None)
        return result[0].values
    
    elif metric == 'sbertscore':
        with open(result_file , 'r') as f:
            raw_data = f.readlines()
            tmp_data = []
            for x in raw_data[1:]:
                try:
                    tmp_data.append(float(x.strip()))
                except:
                    tmp_data.append(float("nan"))
        return np.array(tmp_data)    
    
    else:
        raise ValueError(f"Unknown metric {metric}") 
    

def read_nli_result(file_path):
    with open(file_path, 'r') as f:
        data = f.readlines()
    data = [eval(x.strip()) for x in data]
    data = pd.DataFrame.from_dict(data)
    return data

In [ ]:
gemma_edited = read_outputs('/data/hyeryung/mucoco/outputs/nli/ew9yvzzh/outputs_epsilon0.99.txt.intermediate').drop(columns=['iter0_update'])
gemma_edited_fluency = read_metric_file('/data/hyeryung/mucoco/outputs/nli/ew9yvzzh/results_epsilon0.99-test.txt.fluency', 'fluency')
gemma_original_fluency = read_metric_file('/data/hyeryung/mucoco/new_module/data/logical-consistency/gemma/filtered_0.99_r2-test_500_gemma-2-2b-it_49577.jsonl-results.txt.fluency', 'fluency')
gemma_original_nli = read_nli_result('/data/hyeryung/mucoco/new_module/data/logical-consistency/gemma/filtered_0.99_r2-test_500_gemma-2-2b-it_49577.jsonl-results.txt.nli')
gemma_edited_nli = read_nli_result('/data/hyeryung/mucoco/outputs/nli/ew9yvzzh/results_epsilon0.99-test.txt.nli')

In [ ]:
eda=pd.concat([gemma_edited,gemma_original_nli[['contradiction_prob', 'nli_class']],gemma_edited_nli[['contradiction_prob', 'nli_class']]], axis=1)
eda.columns=['premise','original','located','edited','original_contradiction_prob', 'original_nli_class', 'edited_contradiction_prob', 'edited_nli_class']
eda['original_fluency']=gemma_original_fluency
eda['edited_fluency']=gemma_edited_fluency

In [21]:
eda.to_excel('./gemma_edit.xlsx')

In [22]:
gemma_edited = read_outputs('/data/hyeryung/mucoco/outputs/nli/2eencunf/outputs_epsilon0.99.txt.intermediate').drop(columns=['iter0_update'])
gemma_edited_fluency = read_metric_file('/data/hyeryung/mucoco/outputs/nli/2eencunf/results_epsilon0.99-test.txt.fluency', 'fluency')
gemma_original_fluency = read_metric_file('/data/hyeryung/mucoco/new_module/data/logical-consistency/llama/filtered_0.99_r2-test_500_Llama-3.1-8B-Instruct_49578.jsonl-results.txt.fluency', 'fluency')
gemma_original_nli = read_nli_result('/data/hyeryung/mucoco/new_module/data/logical-consistency/llama/filtered_0.99_r2-test_500_Llama-3.1-8B-Instruct_49578.jsonl-results.txt.nli')
gemma_edited_nli = read_nli_result('/data/hyeryung/mucoco/outputs/nli/2eencunf/results_epsilon0.99-test.txt.nli')

In [23]:
eda=pd.concat([gemma_edited,gemma_original_nli[['contradiction_prob', 'nli_class']],gemma_edited_nli[['contradiction_prob', 'nli_class']]], axis=1)
eda.columns=['premise','original','located','edited','original_contradiction_prob', 'original_nli_class', 'edited_contradiction_prob', 'edited_nli_class']
eda['original_fluency']=gemma_original_fluency
eda['edited_fluency']=gemma_edited_fluency

In [25]:
eda.to_excel('./llama_edit.xlsx')

In [26]:
gemma_edited = read_outputs('/data/hyeryung/mucoco/outputs/nli/aqz7ucpi/outputs_epsilon0.99.txt.intermediate').drop(columns=['iter0_update'])
gemma_edited_fluency = read_metric_file('/data/hyeryung/mucoco/outputs/nli/aqz7ucpi/results_epsilon0.99-test.txt.fluency', 'fluency')
gemma_original_fluency = read_metric_file('/data/hyeryung/mucoco/new_module/data/logical-consistency/phi/filtered_0.99_r2-test_500_Phi-3.5-mini-instruct_49611_postprocess.jsonl-results.txt.fluency', 'fluency')
gemma_original_nli = read_nli_result('/data/hyeryung/mucoco/new_module/data/logical-consistency/phi/filtered_0.99_r2-test_500_Phi-3.5-mini-instruct_49611_postprocess.jsonl-results.txt.nli')
gemma_edited_nli = read_nli_result('/data/hyeryung/mucoco/outputs/nli/aqz7ucpi/results_epsilon0.99-test.txt.nli')

In [28]:
eda=pd.concat([gemma_edited,gemma_original_nli[['contradiction_prob', 'nli_class']],gemma_edited_nli[['contradiction_prob', 'nli_class']]], axis=1)
eda.columns=['premise','original','located','edited','original_contradiction_prob', 'original_nli_class', 'edited_contradiction_prob', 'edited_nli_class']
eda['original_fluency']=gemma_original_fluency
eda['edited_fluency']=gemma_edited_fluency

In [30]:
eda.to_excel('./phi_edit.xlsx')

In [ ]:
## 규인에게 전달해줄 원본 생성 파일

gemma_edited = read_outputs('')
